In [2]:
# Initialize Otter
import otter
grader = otter.Notebook("PW4.ipynb")

# **Лабораторная работа: Математический движок Transformer**
## **Численные методы в фундаменте LLM**

В этой работе мы отойдем от простого анализа исторических данных и перейдем к задачам, с которыми сталкиваются инженеры при разработке и оптимизации больших языковых моделей (LLM). Мы изучим, как классическая вычислительная математика позволяет моделям обрабатывать длинные тексты, эффективно сжиматься и аппроксимировать сложные логические функции.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import lagrange, BarycentricInterpolator, CubicSpline
from scipy.optimize import minimize
import otter
grader = otter.Notebook()

---
## **1. Интерполяция: Проблема расширения контекстного окна**

### **Теоретическая справка**
**С точки зрения вычислительной математики:** Интерполяция на равномерно распределенных узлах часто сталкивается с **феноменом Рунге** — резким ростом амплитуды ошибки на краях интервала при увеличении степени полинома. Узлы Чебышева распределены неравномерно (плотнее у краев), что минимизирует максимальное значение ошибки.

**С точки зрения LLM-инженерии:** Модели обучаются на фиксированной длине текста (например, 512 токенов). Чтобы расширить контекст до 2048 без полного переобучения, инженеры используют **Position Interpolation**. Если интерполяция позиционных эмбеддингов дает выбросы на краях, модель «забывает» начало промпта или начинает галлюцинировать в конце.



### **Вопрос 1: Реализация узлов Чебышева**
**Что нужно сделать:**
1. Рассчитайте 10 узлов Чебышева первого рода для стандартного интервала $[-1, 1]$.
2. Масштабируйте их в интервал $[0, 10]$ по формуле: $x_i = \frac{1}{2}(a+b) + \frac{1}{2}(b-a)\cos(\frac{2i-1}{2n}\pi)$.
3. Создайте интерполятор `BarycentricInterpolator` на основе этих узлов для функции $\sin(x)$.

In [ ]:
# Синтетические данные эмбеддингов (10 известных точек)
x_known = np.linspace(0, 10, 10)
y_known = np.sin(x_known)

k = np.arange(1, 11)
cheb_nodes = np.cos((2*k-1)/(2*10)*np.pi)
cheb_nodes = 5+5*cheb_nodes

poly_cheb = BarycentricInterpolator(cheb_nodes, np.sin(cheb_nodes))

In [19]:
grader.check("q1")

q1 results: All test cases passed!

### **Вопрос 2: Теоретический анализ феномена Рунге**
Объясните, почему стандартная полиномиальная интерполяция на равномерных узлах терпит неудачу на границах контекстного окна и как узлы Чебышева решают эту проблему.

In [21]:
# Напишите ваш ответ между тройными кавычками ниже.
t2_ans = '''
Как было сказано, метод Ньютона использует равномерное распределение узлов вдоль интервала.
Если принять во внимание феномен Рунге, то лучше расставить больше узлов по краям интервала,
что идейно не позволяет сделать метод Ньютона.

Узлы Чебышева, в свою очередь, распределены неравномерно, плотнее по краям интервала.
Это позволяет избежать большой ошибки по границам.
'''
# Записываем в файл для автопроверки
with open("t2.txt", "w", encoding="utf-8") as f:
    f.write(t2_ans)

from IPython.display import display, Markdown
display(Markdown(t2_ans))


Как было сказано, метод Ньютона использует равномерное распределение узлов вдоль интервала.
Если принять во внимание феномен Рунге, то лучше расставить больше узлов по краям интервала,
что идейно не позволяет сделать метод Ньютона.

Узлы Чебышева, в свою очередь, распределены неравномерно, плотнее по краям интервала.
Это позволяет избежать большой ошибки по границам.


In [ ]:
grader.check("q2")

---
## **2. Сплайны: Проектирование функции активации GELU**

### **Теоретическая справка**
**С точки зрения вычислительной математики:** Кубический сплайн — это кусочно-заданная функция, которая сохраняет непрерывность самой функции, её первой и второй производных ($C^2$ непрерывность). Это делает аппроксимацию очень гладкой.

**С точки зрения LLM-инженерии:** Современные модели (BERT, GPT, Llama) используют **GELU (Gaussian Error Linear Unit)**. В отличие от ReLU, GELU дифференцируема во всех точках, что позволяет градиентам течь плавно. Для ускорения работы часто используют табличные значения GELU с последующей сплайн-интерполяцией.



### **Вопрос 3: Расчет производной сплайна**
**Что нужно сделать:**
1. Создайте `CubicSpline` для набора данных `x_nodes` и `y_nodes`.
2. Вычислите градиент (первую производную) в точке $x=0.0$. Это значение критично для предотвращения проблемы «затухающих градиентов».

In [31]:
def gelu_raw(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

x_nodes = np.linspace(-3, 3, 8)
y_nodes = gelu_raw(x_nodes)

cs = CubicSpline(x_nodes, y_nodes)
grad_at_zero = cs(0, 1)

In [32]:
grader.check("q3")

q3 results: All test cases passed!

---
## **3. Оптимизация: Нормы L1 и L2 в задачах сжатия моделей**

### **Теоретическая справка**
**С точки зрения вычислительной математики:** L2-норма (квадратичная) минимизирует среднее отклонение и чувствительна к выбросам. L1-норма (модуль) более устойчива к аномалиям и приводит к разреженным решениям (многие веса становятся равными 0).

**С точки зрения LLM-инженерии:** Веса LLM занимают гигабайты памяти. Чтобы запустить модель на смартфоне, мы используем **прунинг (Pruning)**. Мы заставляем оптимизатор использовать L1-регуляризацию, чтобы «обнулить» маловажные связи в нейросети без потери качества. Подробнее про прунинг можно почитать [тут](https://habr.com/ru/articles/811221/).



### **Вопрос 4: Поиск разреженного решения**
**Что нужно сделать:** Реализуйте две функции потерь для поиска параметров прямой $y = ax + b$. В данных есть один гигантский «выброс». Посмотрите, какой метод (L1 или L2) проигнорирует ошибку и сохранит правильный наклон.

In [36]:
def l2(params, x, y):
  pred = params[0] * x + params[1]
  mse = (pred - y) ** 2
  return np.mean(mse)

def l1(params, x, y):
  pred = params[0] * x + params[1]
  mse = np.abs(pred - y)
  return np.mean(mse)

def f(params, x):
  return params[0] * x + params[1]

weights_x = np.linspace(0, 10, 20)
weights_y = 2 * weights_x + 1
weights_y[-1] = -100 # Мощный шум/выброс

res2 = minimize(l2, [0, 0], args=(weights_x, weights_y))
res1 = minimize(l1, [0, 0], args=(weights_x, weights_y))

l2_slope = res2.x[0]
l1_slope = res1.x[0]

In [37]:
grader.check("q4")

q4 results: All test cases passed!

### **Вопрос 5: Теория прунинга**
Почему минимизация L1-нормы приводит к появлению нулевых весов (разреженности), а L2 — нет? Как это помогает при запуске LLM на мобильных устройствах?

In [50]:
# Напишите ваш ответ между тройными кавычками ниже.
t5_ans = '''
Может, это как-то связано с тем, что в L1-норме мы опускаем вклад членов выше первого порядка,
из-за чего пропускаем данные для изучения поведения функции. В L2-норме мы их учитываем, из-за
чего модель оперирует с большим количеством ненулевых весов. Это заставляет производить более
сложные вычисления, которые могут быть проблематичны для мобильных устройств.

OFFTOP:
Странно, что об этом не сказано ни в этом файле, ни в хендбуке в чате, ни по ссылке выше.
В презентации с лекции просто сказано о существовании этих методов и их конструктивных особенностях,
но это не отвечает на вопрос, почему появляются нулевые веса.

Ответ может быть в корне неверным, потому что я никогда не работал с нейросетями. Мне это не
кажется чем-то интуитивно понятным, поэтому разбираться "на месте" кажется слишком сложным.
Тем более информация по-любому забудется после такого нервозного гуглинга.

Лучше бы лабораторные были в домашнем формате. Так хоть изучаешь информацию осознанно, без спешки.
'''
# Записываем в файл для автопроверки
with open("t5.txt", "w", encoding="utf-8") as f:
    f.write(t5_ans)

from IPython.display import display, Markdown
display(Markdown(t5_ans))


Может, это как-то связано с тем, что в L1-норме мы опускаем вклад членов выше первого порядка,
из-за чего пропускаем данные для изучения поведения функции. В L2-норме мы их учитываем, из-за
чего модель оперирует с большим количеством ненулевых весов. Это заставляет производить более
сложные вычисления, которые могут быть проблематичны для мобильных устройств.

OFFTOP:
Странно, что об этом не сказано ни в этом файле, ни в хендбуке в чате, ни по ссылке выше.
В презентации с лекции просто сказано о существовании этих методов и их конструктивных особенностях,
но это не отвечает на вопрос, почему появляются нулевые веса.

Ответ может быть в корне неверным, потому что я никогда не работал с нейросетями. Мне это не
кажется чем-то интуитивно понятным, поэтому разбираться "на месте" кажется слишком сложным.
Тем более информация по-любому забудется после такого нервозного гуглинга.

Лучше бы лабораторные были в домашнем формате. Так хоть изучаешь информацию осознанно, без спешки.


In [ ]:
grader.check("q5")

---
## **4. Универсальная аппроксимация: Сигмоиды Цыбенко**

### **Теоретическая справка**
**С точки зрения вычислительной математики:** Теорема Цыбенко утверждает, что любая непрерывная функция на компактном множестве может быть аппроксимирована суммой функций вида $\sum \alpha_i \sigma(w_i x + b_i)$.

**С точки зрения LLM-инженерии:** Это фундаментальное обоснование того, почему нейросети способны «выучить» любую логику, если у них достаточно нейронов. В LLM каждый слой — это, по сути, такая сумма (только в матричном виде).



### **Вопрос 5: Скрытый слой вручную**
**Что нужно сделать:** Напишите код функции, которая вычисляет выход нейронного слоя. Параметры передаются как один плоский массив `params`, который нужно правильно «распаковать» на веса, смещения и коэффициенты суммы.

In [48]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def manual_neural_layer(x, params):
    """
    params: np.array вида [alpha1...alphaN, w1...wN, b1...bN]
    """
    N = params.size // 3
    alpha = params[:N]
    weight = params[N:2*N]
    b = params[2*N:3*N]
    return [np.sum(alpha * sigmoid(weight * x + b))]

In [49]:
grader.check("q6")

q6 results: All test cases passed!

---

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [52]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True, pdf=False)

Running your submission against local test cases...


Your submission received the following results when run against available test cases:


